# Random substitution baseline

This notebook implements the random local-editing baseline used in the molecular-editing benchmark.

## Purpose

For each fixed parent molecule, generate up to `N_CANDIDATES_PER_PARENT` unique valid molecules using random local structural edits. The generation procedure is **descriptor-agnostic**: HBD, HBA, aromatic rings, and rotatable bonds are not used to guide generation.

Descriptor changes are calculated only **after generation**. This makes the method a neutral random baseline against which directed molecular-editing methods can be compared.

## Preparation

### Import library

In [1]:
import random
import sys

import pandas as pd
from tqdm import tqdm

from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")

sys.path.append("..")

from configs.benchmark_config import (
    N_CANDIDATES_PER_PARENT,
    RANDOM_SEED,
)

from utils.parents import prepare_parent
from utils.records import append_candidate_record

random.seed(RANDOM_SEED)

### Import parent compounds

In [2]:
df = pd.read_csv("../data/chembl_1000_parents.csv")
df.head()

,chembl_id,smiles,mw,hbd,hba,logp,rotb,ar
0,CHEMBL85118,COC(=O)CCCCCC1=N/C(=C\c2[nH]c(-c3ccc[nH]3)cc2O...,367.449,2,4,4.49360,9,2
1,CHEMBL91137,CCCc1nnc([S+]([O-])Cc2ncc(C)c(OC)c2C)o1,309.391,0,6,2.35034,6,2
2,CHEMBL441229,C=C1C(O)C(O)C(O)C(O)C1O,176.168,5,5,-2.63930,0,0
3,CHEMBL1068,NC(=O)N1c2ccccc2CC(=O)c2ccccc21,252.273,1,2,2.64220,0,2
4,CHEMBL105373,COc1cc(-n2sc3ncccc3c2=O)cc(OC)c1OC,318.354,0,6,2.47300,4,3


## Compounds generation: Random local editing

In [3]:
ATOM_TYPES = [6, 7, 8, 9, 16, 17]

def random_edit(mol):
    try:
        mol = Chem.RWMol(mol)
        edit = random.choice(["replace", "add", "remove"])

        if edit == "replace":
            idx = random.randrange(mol.GetNumAtoms())
            old = mol.GetAtomWithIdx(idx).GetAtomicNum()
            choices = [x for x in ATOM_TYPES if x != old]
            mol.GetAtomWithIdx(idx).SetAtomicNum(random.choice(choices))

        elif edit == "add":
            idx = random.randrange(mol.GetNumAtoms())
            new_idx = mol.AddAtom(Chem.Atom(random.choice(ATOM_TYPES)))
            mol.AddBond(idx, new_idx, Chem.BondType.SINGLE)

        else:
            terminal = [
                a.GetIdx()
                for a in mol.GetAtoms()
                if a.GetDegree() == 1
            ]
            if not terminal:
                return None
            mol.RemoveAtom(random.choice(terminal))

        new_mol = mol.GetMol()
        Chem.SanitizeMol(new_mol)

        return new_mol

    except Exception:
        return None

In [4]:
records = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    prepared = prepare_parent(row)
    if prepared is None:
        continue

    chembl_id, parent_smiles, parent_desc = prepared
    parent = Chem.MolFromSmiles(parent_smiles)

    generated_smiles = set()

    for _ in range(N_CANDIDATES_PER_PARENT):
        new_mol = random_edit(parent)
        if new_mol is None:
            continue

        new_smiles = Chem.MolToSmiles(
            new_mol,
            canonical=True,
            isomericSmiles=True,
        )

        if (
            new_smiles == parent_smiles
            or new_smiles in generated_smiles
        ):
            continue

        generated_smiles.add(new_smiles)

    for new_smiles in generated_smiles:
        append_candidate_record(
            records=records,
            chembl_id=chembl_id,
            parent_smiles=parent_smiles,
            generated_smiles=new_smiles,
            parent_desc=parent_desc,
        )

df_random = pd.DataFrame(records)

print(df_random.shape)
df_random.head()

  0%|          | 0/1000 [00:00<?, ?it/s]

100%|██████████| 1000/1000 [00:40<00:00, 24.83it/s]


(30248, 15)


,chembl_id,parent_smiles,generated_smiles,hbd_parent,hbd_generated,delta_hbd,hba_parent,hba_generated,delta_hba,ar_parent,ar_generated,delta_ar,rotb_parent,rotb_generated,delta_rotb
0,CHEMBL85118,COC(=O)CCCCCC1=N/C(=C\c2[nH]c(-c3ccc[nH]3)cc2O...,COC(=O)CCC(S)CCC1=N/C(=C\c2[nH]c(-c3ccc[nH]3)c...,2,3,1,4,5,1,2,2,0,9,9,0
1,CHEMBL85118,COC(=O)CCCCCC1=N/C(=C\c2[nH]c(-c3ccc[nH]3)cc2O...,COc1cc(-c2ccc[nH]2)[nH]c1/C=C1/C=CC(CCCCCC(=O)...,2,3,1,4,5,1,2,2,0,9,9,0
2,CHEMBL85118,COC(=O)CCCCCC1=N/C(=C\c2[nH]c(-c3ccc[nH]3)cc2O...,COC(=O)CCCCCC1=N/C(=C\c2[nH]c(-c3ccc[nH]3)cc2O...,2,3,1,4,5,1,2,2,0,9,9,0
3,CHEMBL85118,COC(=O)CCCCCC1=N/C(=C\c2[nH]c(-c3ccc[nH]3)cc2O...,COC(=O)CCCCCC1=N/C(=C\c2[nH]c(-c3ccc[nH]3)cc2O...,2,3,1,4,5,1,2,2,0,9,9,0
4,CHEMBL85118,COC(=O)CCCCCC1=N/C(=C\c2[nH]c(-c3ccc[nH]3)cc2O...,CCOC(=O)CCCCCC1=N/C(=C\c2[nH]c(-c3ccc[nH]3)cc2...,2,2,0,4,4,0,2,2,0,9,10,1


## 4. Save generated products

The output columns intentionally match the Random substitution output so that all methods can be evaluated with the same downstream analysis code.


In [5]:
OUTPUT_FILE = "../results/random_local_editing.csv"

df_random.to_csv(OUTPUT_FILE, index=False)
print(f"Saved: {OUTPUT_FILE}")

Saved: ../results/random_local_editing.csv


## Output and downstream evaluation

- `results/random_local_editing.csv` — product-level random local editing results.

The common analysis notebook should evaluate the same criteria for every method:

- HBD increase/decrease: `delta_hbd == ±1`
- HBA increase/decrease: `delta_hba == ±1`
- Aromatic-ring increase/decrease: `delta_ar == ±1`
- Rotatable-bond increase/decrease: `delta_rotb == ±1`
- Selective HBD increase: `delta_hbd == +1` and `delta_hba == 0`